# ShopGuard URL Fraud Detection — Model Training (merged_v1)

**Goal**: Train an MLP classifier on 16 URL-derived features to detect phishing URLs, then export to ONNX for `ai-worker` inference.

**Inputs**: `merged_v1.parquet` (≈111k rows; URL + label, no precomputed features)
**Outputs** (saved to Drive folder `url_model_artifacts/`):
- `fraud_model.onnx` — trained model
- `scaler.pkl` — input normalization
- `feature_columns.json` — feature order (CRITICAL: ai-worker must use this exact order)
- `model_metadata.json` — training info, metrics, threshold

**Pipeline**: load parquet → **extract 16 URL features** → split → scale → train (PyTorch MLP) → evaluate → export ONNX → sanity check.

**Label convention** (matches `merged_v1.parquet` / existing `url_classifier.py`):
- `0 = phishing`
- `1 = legitimate`

**Feature extraction parity**: the 16-feature extraction logic in this notebook MUST match `ai-worker/worker/pipeline/url_classifier.py` exactly, including the documented dataset extraction quirks (e.g. `query_param_count` is 1-based, `tld_length` includes port, etc.). Do not 'fix' these — the model is trained on values produced by this buggy logic and inference must reproduce them.

## 1. Environment Setup

In [ ]:
# Colab — ONNX 관련 패키지만 추가 설치 (PyTorch, sklearn, pandas, pyarrow는 기본 제공)
!pip install -q onnx onnxruntime onnxscript

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import joblib

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load Data

Mount Google Drive and load `merged_v1.parquet`. We only need `url`, `label`, and `source` columns — the 16 URL features are computed in the next cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PARQUET_PATH = '/content/drive/MyDrive/26S_CS350_Software_Engineering/dataset/merged_v1.parquet'

df = pd.read_parquet(PARQUET_PATH, columns=['url', 'label', 'source'])
print(f"Shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts(dropna=False)}")
print(f"\nSource distribution:\n{df['source'].value_counts(dropna=False)}")
print(f"\nNull url count: {df['url'].isna().sum()}")
df.head()

## 3. Feature Extraction (16 URL features)

These functions are **copied verbatim** from `ai-worker/worker/pipeline/url_classifier.py`. They reproduce the LegitPhish dataset's original (buggy) extraction logic so that the model trained here is compatible with the existing ai-worker inference code. They are verified to reproduce `url_features_extracted1.csv` to ≈99.997% (the residual rows are malformed/mislabeled URLs in the dataset itself).

Known quirks preserved on purpose:
- `query_param_count` = `len(urlparse(url).query.split('&'))`; an empty query yields `['']` → **1**, so the count is effectively 1-based.
- `token_count` splits on `. / ? = &` (the `&` delimiter matters for multi-param URLs).
- `tld_length` / `tld_popularity` use `netloc.split('.')[-1]`, which **includes the port** (e.g. `com:8080`), so a ported host is never "popular".
- IP-address hostnames are parsed like regular domains (last octet treated as TLD).
- `subdomain_count` = `len(netloc.split('.')) - 2`: does not handle compound TLDs (`.co.kr`, `.co.uk`), and a **scheme-less URL** (empty netloc) yields `['']` → **-1**.
- `suspicious_file_extension` matches an extension at the **end of the URL path** against `{exe, zip, bin, apk, cmd, js}`.

If you change any of these, you MUST also update `url_classifier.py` and re-test with `url_model_artifacts/_validate.py`.

In [ ]:
import math
import ipaddress
import re
from urllib.parse import urlparse
from collections import Counter


def _get_tld_true(url):
    parsed = urlparse(url)
    domain = parsed.netloc.split(':')[0]
    return domain.split('.')[-1]


def _get_tld(url):
    parsed = urlparse(url)
    return parsed.netloc.split('.')[-1]


def _url_length(url):
    return len(url)


def _has_ip_address(url):
    parsed = urlparse(url)
    domain_ = parsed.netloc.split(':')[0]
    try:
        ipaddress.ip_address(domain_)
        return 1
    except ValueError:
        return 0


def _dot_count(url):
    return url.count('.')


def _https_flag(url):
    return int(url.startswith('https://'))


def _url_entropy(url):
    if not url:
        return 0.0
    frequencies = Counter(url)
    n = len(url)
    entropy = 0.0
    for count in frequencies.values():
        p = count / n
        entropy -= p * math.log2(p)
    return entropy


def _token_count(url):
    delimiters = r"\.|\/|\?|\=|\&"
    return len(re.split(delimiters, url))


def _subdomain_count(url):
    parsed = urlparse(url)
    parts = parsed.netloc.split('.')
    # Empty netloc (scheme-less URL) -> [''] -> 1 - 2 == -1 (kept to match dataset).
    return len(parts) - 2


def _query_param_count(url):
    # Query string split on '&'; empty query -> [''] -> 1 (matches dataset).
    return len(urlparse(url).query.split('&'))


def _tld_length(url):
    return len(_get_tld(url))


def _path_length(url):
    return len(urlparse(url).path)


def _has_hyphen_in_domain(url):
    return int('-' in urlparse(url).netloc)


def _number_of_digits(url):
    return sum(c.isdigit() for c in url)


_POPULAR_TLDS = {"com", "org", "net", "edu", "gov"}


def _tld_popularity(url):
    # Port-inclusive TLD ("com:8080" is NOT popular), matching dataset.
    return int(_get_tld(url).lower() in _POPULAR_TLDS)


_SUSPICIOUS_EXTS = {"exe", "zip", "bin", "apk", "cmd", "js"}


def _suspicious_file_extension(url):
    # Match the extension at the END of the URL path only.
    path = urlparse(url).path
    m = re.search(r"\.([A-Za-z0-9]+)$", path)
    return int(bool(m and m.group(1).lower() in _SUSPICIOUS_EXTS))


def _domain_name_length(url):
    parsed = urlparse(url)
    parts = parsed.netloc.split('.')
    if len(parts) < 2:
        return 0
    return len(parts[-2])


def _percentage_numeric_chars(url):
    digits = _number_of_digits(url)
    return 100 * digits / len(url) if url else 0.0


FEATURE_COLUMNS = [
    'url_length', 'has_ip_address', 'dot_count', 'https_flag',
    'url_entropy', 'token_count', 'subdomain_count', 'query_param_count',
    'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits',
    'tld_popularity', 'suspicious_file_extension', 'domain_name_length',
    'percentage_numeric_chars',
]

_EXTRACTORS = {
    'url_length': _url_length,
    'has_ip_address': _has_ip_address,
    'dot_count': _dot_count,
    'https_flag': _https_flag,
    'url_entropy': _url_entropy,
    'token_count': _token_count,
    'subdomain_count': _subdomain_count,
    'query_param_count': _query_param_count,
    'tld_length': _tld_length,
    'path_length': _path_length,
    'has_hyphen_in_domain': _has_hyphen_in_domain,
    'number_of_digits': _number_of_digits,
    'tld_popularity': _tld_popularity,
    'suspicious_file_extension': _suspicious_file_extension,
    'domain_name_length': _domain_name_length,
    'percentage_numeric_chars': _percentage_numeric_chars,
}


def extract_features(url):
    return [_EXTRACTORS[c](url) for c in FEATURE_COLUMNS]


print(f"Defined {len(FEATURE_COLUMNS)} feature extractors.")
# Sanity check on a couple of well-known URLs
for u in ['https://www.coupang.com/vp/products/123?vendorItemId=456',
          'http://192.210.150.19/eTzMQwJ134.bin',
          'https://en.wikipedia.org/wiki/Phishing']:
    feats = dict(zip(FEATURE_COLUMNS, extract_features(u)))
    print(f"{u}\n  {feats}")

In [ ]:
# Drop rows with null url or label, then extract features in bulk.
before = len(df)
df = df.dropna(subset=['url', 'label']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows missing url/label. Remaining: {len(df):,}")

# Some legitphish URLs may be malformed; surface any extraction errors instead of silently NaN-ing.
def safe_extract(u):
    try:
        return extract_features(u)
    except Exception:
        return None

raw_feats = df['url'].map(safe_extract)
n_failed = raw_feats.isna().sum()
if n_failed:
    print(f"WARN: {n_failed} URLs failed feature extraction (will be dropped).")
    df = df[raw_feats.notna()].reset_index(drop=True)
    raw_feats = raw_feats[raw_feats.notna()].reset_index(drop=True)

X = np.asarray(raw_feats.tolist(), dtype=np.float32)
y = df['label'].astype(np.float32).values
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Positive ratio (legitimate=1): {y.mean():.4f}")
print(f"Per-source counts:\n{df['source'].value_counts()}")

## 4. Train/Val/Test Split

70/15/15, stratified by label.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED  # 0.15/0.85 ≈ 0.1765
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train pos ratio: {y_train.mean():.4f}")
print(f"Val pos ratio: {y_val.mean():.4f}")
print(f"Test pos ratio: {y_test.mean():.4f}")

## 5. Feature Scaling

`StandardScaler` is fit on train only. The scaler is saved alongside the model — inference at ai-worker must reuse this exact scaler or the model sees out-of-distribution inputs.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaler mean (first 5):", scaler.mean_[:5])
print("Scaler std (first 5):", scaler.scale_[:5])

## 6. Model Definition

Simple MLP `16 → 64 → 32 → 1`, dropout 0.3. Output is the raw logit (sigmoid is part of `BCEWithLogitsLoss`).
Identical architecture to the previous URL classifier so the ONNX drop-in works without changing ai-worker inference code.

In [ ]:
class FraudMLP(nn.Module):
    def __init__(self, in_dim=16, hidden1=64, hidden2=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x)

model = FraudMLP(in_dim=len(FEATURE_COLUMNS)).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal params: {total_params}")

## 7. Training Loop

- Loss: `BCEWithLogitsLoss` with `pos_weight` (= neg_count / pos_count, class imbalance correction)
- Optimizer: Adam, lr=1e-3
- Batch size: 256, max 30 epochs, early stop after 5 epochs without val_loss improvement

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

X_train_t = torch.from_numpy(X_train_scaled)
y_train_t = torch.from_numpy(y_train).unsqueeze(1)
X_val_t = torch.from_numpy(X_val_scaled).to(device)
y_val_t = torch.from_numpy(y_val).unsqueeze(1).to(device)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256, shuffle=True,
)

pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
pos_weight = torch.tensor([neg_count / pos_count], device=device)
print(f"pos_weight: {pos_weight.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 30
PATIENCE = 5
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_preds = (torch.sigmoid(val_logits) > 0.5).float()
        val_acc = (val_preds == y_val_t).float().mean().item()

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
model.eval()
print(f"\nBest val_loss: {best_val_loss:.4f}")

## 8. Evaluation

Test set metrics. For phishing detection, **recall on the phishing class (label=0)** matters most — missing a real scam is the high-cost error.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

X_test_t = torch.from_numpy(X_test_scaled).to(device)

with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()

THRESHOLD = 0.5
test_preds = (test_probs > THRESHOLD).astype(np.int32)
y_test_int = y_test.astype(np.int32)

print(f"Threshold: {THRESHOLD}")
print(f"Accuracy:  {accuracy_score(y_test_int, test_preds):.4f}")
print(f"Precision: {precision_score(y_test_int, test_preds):.4f}")
print(f"Recall:    {recall_score(y_test_int, test_preds):.4f}")
print(f"F1:        {f1_score(y_test_int, test_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_int, test_probs):.4f}")

print(f"\nConfusion matrix:\n{confusion_matrix(y_test_int, test_preds)}")
print(f"\nClassification report:\n{classification_report(y_test_int, test_preds, target_names=['phishing', 'legitimate'])}")

# Per-source breakdown (e.g. how well does the model do on phishpedia URLs separately?)
test_idx = np.arange(len(X))[-len(X_test):]  # not a guaranteed slice; recompute via mask below
# proper per-source metrics using the original df alignment:
_X_temp, _X_test, _y_temp, _y_test, _src_temp, _src_test = train_test_split(
    X, y, df['source'].values, test_size=0.15, stratify=y, random_state=SEED
)
for src in pd.Series(_src_test).unique():
    mask = (_src_test == src)
    if mask.sum() == 0:
        continue
    y_s = _y_test[mask].astype(np.int32)
    # recompute predictions on the same test rows via cache from above (probs already match X_test order)
    p_s = test_probs[mask]
    pred_s = (p_s > THRESHOLD).astype(np.int32)
    print(f"\n[source={src}] n={mask.sum():,} | acc={accuracy_score(y_s, pred_s):.4f} | "
          f"precision={precision_score(y_s, pred_s, zero_division=0):.4f} | "
          f"recall={recall_score(y_s, pred_s, zero_division=0):.4f}")

## 9. Export to ONNX

Save 4 artifacts to Drive. These are the files ai-worker reads at startup.

In [ ]:
import os
from datetime import datetime

OUTPUT_DIR = '/content/drive/MyDrive/26S_CS350_Software_Engineering/url_model_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. ONNX export
model.eval()
model_cpu = model.to('cpu')
dummy_input = torch.randn(1, len(FEATURE_COLUMNS), dtype=torch.float32)

onnx_path = os.path.join(OUTPUT_DIR, 'fraud_model.onnx')
torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['features'],
    output_names=['logit'],
    dynamic_axes={
        'features': {0: 'batch_size'},
        'logit': {0: 'batch_size'},
    },
)
print(f"OK ONNX saved: {onnx_path}")

# 2. Scaler
scaler_path = os.path.join(OUTPUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"OK Scaler saved: {scaler_path}")

# 3. Feature columns
feature_path = os.path.join(OUTPUT_DIR, 'feature_columns.json')
with open(feature_path, 'w') as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)
print(f"OK Feature columns saved: {feature_path}")

# 4. Metadata
metadata = {
    'trained_at': datetime.now().isoformat(),
    'framework': f'PyTorch {torch.__version__}',
    'model_arch': 'MLP (16 -> 64 -> 32 -> 1)',
    'training_dataset': 'merged_v1.parquet',
    'dataset_sources': df['source'].value_counts().to_dict(),
    'label_convention': {'0': 'phishing', '1': 'legitimate'},
    'n_features': len(FEATURE_COLUMNS),
    'n_train': int(len(X_train)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test)),
    'threshold': float(THRESHOLD),
    'metrics': {
        'accuracy': float(accuracy_score(y_test_int, test_preds)),
        'precision': float(precision_score(y_test_int, test_preds)),
        'recall': float(recall_score(y_test_int, test_preds)),
        'f1': float(f1_score(y_test_int, test_preds)),
        'roc_auc': float(roc_auc_score(y_test_int, test_probs)),
    },
}
metadata_path = os.path.join(OUTPUT_DIR, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"OK Metadata saved: {metadata_path}")
print(f"\nMetadata:\n{json.dumps(metadata, indent=2, default=str)}")

## 10. Sanity Check

Reload the ONNX model and verify PyTorch and ONNX outputs match within numerical tolerance. **Do not ship the artifacts unless this cell passes.**

In [ ]:
import onnxruntime as ort

ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

sample_idx = np.random.choice(len(X_test_scaled), 10, replace=False)
sample_x = X_test_scaled[sample_idx]

with torch.no_grad():
    torch_logits = model_cpu(torch.from_numpy(sample_x)).numpy()

ort_inputs = {'features': sample_x}
ort_logits = ort_session.run(None, ort_inputs)[0]

print("PyTorch logits:", torch_logits.flatten())
print("ONNX logits:   ", ort_logits.flatten())
print(f"\nMax abs diff: {np.abs(torch_logits - ort_logits).max():.2e}")

assert np.allclose(torch_logits, ort_logits, atol=1e-5), "ONNX and PyTorch outputs differ!"
print("\nOK Sanity check passed. Artifacts safe to ship to ai-worker.")

## 11. ai-worker Integration Notes

Replace the contents of `ai-worker/worker/pipeline/url_model_artifacts/` with the 4 files just written to Drive:
- `fraud_model.onnx`
- `scaler.pkl`
- `feature_columns.json`
- `model_metadata.json`

**Score direction** (unchanged from previous model): `url_classifier.compute_url_risk_score(url)` returns `(1 - P_legit) * 100`. Since the merged-dataset label is `1=legitimate`, the model output `P(label=1) = P_legit`, and the fraud score formula stays correct.

**Validation before deploy**: run `ai-worker/worker/pipeline/url_model_artifacts/_validate.py` (existing script) to confirm the new ONNX produces the same outputs as the Colab PyTorch model on a fixed sample set.